# Chicell2cell Pipeline

**Section 1** — Preprocessing (runs fresh for demonstration and visualisation only)

**Section 2 onwards** — Loads your already processed data for all real analysis

In [ ]:
import os
os.chdir('/Users/chigozie/Downloads/chicell2cell')
print(os.getcwd())

In [ ]:
import os
print(os.listdir('DATA/'))

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import torch
import pickle
import chicell2cell as c2c
from chicell2cell.model import (
    FLAGS, CELL2CELLGATVAE, train_model,
    mask_test_edges_general_link_prediction,
    normalize_data, adj_to_edge_index, ThresholdSelector,
    evaluate_on_edges, compute_scores, get_link_predictions_decoder,
    load_model, HyperparameterTuner,
)
sc.settings.set_figure_params(figsize=(14,10), dpi=60, facecolor='white', fontsize=16)
device = torch.device('cpu')
torch.manual_seed(42)
np.random.seed(42)
print('chicell2cell version:', c2c.__version__)

---
## SECTION 1 — PREPROCESSING
#### For demonstration and visualisation only. NOT used for model training.
---

In [ ]:
adata_raw = sc.read_h5ad('DATA/Human_Glioblastoma_Whole_10xvisium.h5ad')
print(f'Raw data: {adata_raw.shape}')

In [ ]:
adata_raw = c2c.preprocessing.run_qc(adata_raw, plot=True)

In [ ]:
adata_raw = c2c.preprocessing.filter_cells_genes(
    adata_raw, min_genes=500, min_cells=3, max_pct_mito=10.0, plot=True
)
print(f'After filtering: {adata_raw.shape}')

In [ ]:
adata_raw = c2c.preprocessing.remove_doublets(adata_raw)

In [ ]:
adata_raw = c2c.preprocessing.normalize_and_log(adata_raw)

In [ ]:
adata_raw = c2c.preprocessing.select_hvg_and_cluster(
    adata_raw, min_disp=0.5, min_mean=0.0125, max_mean=3.0,
    span=0.3, n_bins=20, flavor='seurat',
    n_pcs=31, n_neighbors=15, leiden_resolution=0.39, plot=True,
)

In [ ]:
cluster_to_cell_type = {
    '0': 'NPC-like-Tumor', '1': 'Microglia/Macrophages',
    '2': 'OPC-like-tumor',  '3': 'AC-like',
    '4': 'Endothelial',     '5': 'Inhibition_neurons',
    '6': 'MES-like',        '7': 'Astrocyte',
    '8': 'Neurons',
}
adata_raw = c2c.preprocessing.annotate_cell_types(
    adata_raw, cluster_to_cell_type=cluster_to_cell_type, plot=True
)
print('Preprocessing demo complete — plots shown above')
print('NOTE: adata_raw is NOT used for downstream analysis')

---
## SECTION 2 — LOAD ALREADY PROCESSED DATA
#### Used for ALL downstream analysis
---

In [ ]:
adata_lr_filtered = sc.read_h5ad('DATA/adata_lr_filtered.h5ad')
lr_pairs = pd.read_csv('ligand_receptor/CellChatDB_human_ligand_receptor_signaling_core.csv')
genes = set(adata_lr_filtered.var_names)
lr_pairs_filtered = lr_pairs[
    lr_pairs['ligand'].isin(genes) & lr_pairs['receptor'].isin(genes)
].copy()
print(f'adata_lr_filtered : {adata_lr_filtered.shape}')
print(f'cell_type present : {"cell_type" in adata_lr_filtered.obs.columns}')
print(f'LR pairs          : {len(lr_pairs_filtered)}')
print(adata_lr_filtered.obs['cell_type'].value_counts())

---
## SECTION 3 — GRAPH CONSTRUCTION
---

In [ ]:
cell_adj, threshold = c2c.graph.build_cell_graph(adata_lr_filtered, k=15, percentile=90)
c2c.visualization.plot_spatial_graph(adata_lr_filtered, cell_adj)

In [ ]:
gene_adj, n_edges = c2c.graph.load_geneadj(adata_lr_filtered, lr_pairs_filtered)
print(f'Gene graph: {gene_adj.shape}, edges: {n_edges}')

---
## SECTION 4 — PREPARE TRAINING DATA
---

In [ ]:
cell_adj_array = cell_adj.toarray() if hasattr(cell_adj,'toarray') else cell_adj
gene_adj_array = gene_adj.values
(
    adj_cell,
    train_edges_cell, train_edges_false_cell,
    val_edges_cell,   val_edges_false_cell,
    test_edges_cell,  test_edges_false_cell,
) = mask_test_edges_general_link_prediction(cell_adj_array, FLAGS.prop_test, FLAGS.prop_val)
(
    adj_gene,
    train_edges_gene, train_edges_false_gene,
    val_edges_gene,   val_edges_false_gene,
    test_edges_gene,  test_edges_false_gene,
) = mask_test_edges_general_link_prediction(gene_adj_array, FLAGS.prop_test, FLAGS.prop_val)
X = adata_lr_filtered.X.toarray() if hasattr(adata_lr_filtered.X,'toarray') else adata_lr_filtered.X
cell_features = torch.FloatTensor(normalize_data(X)).to(device)
gene_features = torch.FloatTensor(normalize_data(adata_lr_filtered.var[['n_cells']].values)).to(device)
cell_edge_index = adj_to_edge_index(adj_cell).to(device)
gene_edge_index = adj_to_edge_index(adj_gene).to(device)
exp = pd.DataFrame(X)
num_nodes_cell = adj_cell.shape[0]
num_nodes_gene = adj_gene.shape[0]
pos_weight_cell = float(adj_cell.shape[0]**2 - adj_cell.sum()) / adj_cell.sum()
norm_cell = adj_cell.shape[0]**2 / float((adj_cell.shape[0]**2 - adj_cell.sum()) * 2)
pos_weight_gene = float(adj_gene.shape[0]**2 - adj_gene.sum()) / adj_gene.sum()
norm_gene = adj_gene.shape[0]**2 / float((adj_gene.shape[0]**2 - adj_gene.sum()) * 2)
print(f'Cell features : {cell_features.shape}')
print(f'Gene features : {gene_features.shape}')
print('Data preparation complete')

---
## SECTION 5 — LOAD YOUR ALREADY TRAINED MODEL
---

In [ ]:
from chicell2cell.model import load_model

best_run = load_model('saved_model/m_best_run.pkl')

b_model   = best_run['model']
b_history = best_run['history']
b_epoch   = best_run['best_epoch']
b_params  = best_run['params']

print(f'Model loaded. Best epoch: {b_epoch}')
print(f'Params: {b_params}')

---
## SECTION 6 — TRAINING PLOTS AND EVALUATION
---

In [ ]:
c2c.visualization.plot_training_history(b_history)

In [ ]:
test_eval = evaluate_on_edges(
    b_model, cell_features, cell_edge_index, gene_features, gene_edge_index,
    test_edges_cell, test_edges_false_cell, test_edges_gene, test_edges_false_gene,
)
print(f"Test Cell ROC: {test_eval['roc_cell']:.4f}  AP: {test_eval['ap_cell']:.4f}")
print(f"Test Gene ROC: {test_eval['roc_gene']:.4f}  AP: {test_eval['ap_gene']:.4f}")

---
## SECTION 7 — EXTRACT MODEL PREDICTIONS
---

In [ ]:
b_model.eval()
with torch.no_grad():
    outputs = b_model(cell_features, cell_edge_index, gene_features, gene_edge_index)
cell_embeddings   = outputs['z_mean_cell'].cpu().numpy()
decoder_pred      = outputs['adj_pred_cell'].cpu().numpy()
attention_weights = b_model.cell_encoder.attention_weights.cpu().numpy()
edge_index_att    = b_model.cell_encoder.edge_index_with_attention.cpu().numpy()
print(f'Decoder pred shape      : {decoder_pred.shape}')
print(f'Attention weights shape : {attention_weights.shape}')

In [ ]:
selector = ThresholdSelector(test_edges_cell, test_edges_false_cell)
adj_pred, adj_binary, all_acc, max_acc, optimal_threshold = selector.select(
    b_model, cell_features, cell_edge_index, gene_features, gene_edge_index
)
test_roc, test_ap = compute_scores(test_edges_cell, test_edges_false_cell, cell_embeddings)
print(f'ROC-AUC   : {test_roc:.4f}')
print(f'AP        : {test_ap:.4f}')
print(f'Accuracy  : {max_acc:.4f}')
print(f'Threshold : {optimal_threshold:.4f}')

In [ ]:
# Score distribution — positive vs negative communications
pos_preds, neg_preds = c2c.visualization.plot_score_distribution(
    b_model, cell_features, cell_edge_index, gene_features, gene_edge_index,
    test_edges_pos=test_edges_cell, test_edges_neg=test_edges_false_cell,
)

---
## SECTION 8 — CLUSTER COMMUNICATION
---

In [ ]:
communication_scores_df = c2c.communication.cluster_communication(
    adata=adata_lr_filtered, decoder_pred=decoder_pred,
    attention_weights=attention_weights, edge_index_att=edge_index_att,
    database_name='chicell2cell', clustering='cell_type',
    attention_threshold=0.0, n_permutations=500, random_seed=42, copy=False,
)
print(communication_scores_df.head(10))

---
## SECTION 9 — SIGNIFICANT LR COMMUNICATIONS
---

In [ ]:
uns_key = 'chicell2cell_cluster-cell_type-total-total'
significant_lr_comms = c2c.communication.extract_significant_lr_communications(
    adata=adata_lr_filtered, lr_pairs=lr_pairs_filtered,
    attention_weights=attention_weights, edge_index_att=edge_index_att,
    uns_key=uns_key, clustering='cell_type', p_value_cutoff=0.05,
)
filtered_lr = significant_lr_comms[
    ((significant_lr_comms['ligand_expression'] > 0) |
     (significant_lr_comms['receptor_expression'] > 0)) &
    (significant_lr_comms['annotation'] == 'Secreted Signaling')
].copy()
filtered_lr.to_csv('filtered_lr_cell_cell_comms.csv', index=False)
print(f'Significant interactions: {len(filtered_lr)}')

---
## SECTION 10 — COMMUNICATION PATTERNS (6-panel)
---

In [ ]:
c2c.visualization.plot_communication_patterns(
    filtered_lr, source_cell='Inhibition_neurons',
)

---
## SECTION 11 — SPECTRAL CLUSTERING (7-panel)
---

In [ ]:
adata_clustered, embeddings = c2c.visualization.spectral_clustering_analysis(
    adata_lr_filtered, decoder_pred, k=9
)
c2c.visualization.plot_seven_panels(adata_clustered, embeddings)

---
## SECTION 12 — JACCARD COMPARISON
---

In [ ]:
import glob
empty = pd.DataFrame(columns=['source','target','ligand','receptor'])
all_db = {}
for f in sorted(glob.glob('database_analysis/*_results.csv')):
    name = os.path.basename(f).replace('_results.csv','')
    df = pd.read_csv(f)
    if 'source' in df.columns:
        df = df[df['source'] != df['target']].reset_index(drop=True)
    all_db[name] = df
    print(f'  {name}: {len(df):,}')
chicelldb = significant_lr_comms[['source','target','ligand','receptor']]
databases, jac_matrix, key_presence, jac_vs_chicell = \
    c2c.comparison.analyze_ccc_jaccard_vs_chicell(
        chicelldb=chicelldb,
        cellphonedb=all_db.get('cellphonedb', empty),
        connectomedb2020=all_db.get('connectomedb2020', empty),
        celltalkdb=all_db.get('celltalkdb', empty),
        icellnet=all_db.get('icellnet', empty),
        italk=all_db.get('italk', empty),
        cellcall=all_db.get('cellcall', empty),
        cellinker=all_db.get('cellinker', empty),
        include_cells=False, max_items_for_heatmap=100,
    )

---
## SECTION 13 — VENN + UPSET PLOT
---

In [ ]:
from chicell2cell.comparison import plot_venn_upset
plot_venn_upset(
    chicelldb=significant_lr_comms[['source','target','ligand','receptor']],
    cellphonedb_results=all_db.get('cellphonedb', empty),
    connectomedb2020_results=all_db.get('connectomedb2020', empty),
    celltalkdb_results=all_db.get('celltalkdb', empty),
    icellnet_results=all_db.get('icellnet', empty),
    italk_results=all_db.get('italk', empty),
    cellcall_results=all_db.get('cellcall', empty),
    cellinker_results=all_db.get('cellinker', empty),
)